# ChatSentry — Deploy on Chameleon Cloud

This notebook provisions a KVM@TACC VM, installs Docker, clones the ChatSentry repo,
and starts the full data pipeline stack (PostgreSQL, MinIO, FastAPI).

## Prerequisites
- Chameleon account + project membership
- SSH key uploaded to KVM@TACC
- HuggingFace API token

## Architecture
| Service | Port | Purpose |
|---------|------|---------|
| PostgreSQL | 5432 | Application state (users, messages, flags, moderation) |
| MinIO | 9000/9001 | Object storage (raw messages, training data) |
| FastAPI | 8000 | Inference API (/messages, /flags, /health) |
| Adminer | 5050 | PostgreSQL browser UI |

In [ ]:
# run in Chameleon Jupyter environment
from chi import server, context, lease, network
import chi, os, time, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv('USER')

: 

## Step 1: Create Lease

Reserve an `m1.xlarge` VM for 8 hours.

In [ ]:
# run in Chameleon Jupyter environment
l = lease.Lease("proj09_Data", duration=datetime.timedelta(hours=8))
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.xlarge"), amount=1)
l.submit(idempotent=True)

In [ ]:
# run in Chameleon Jupyter environment
l.show()

## Step 2: Launch VM Instance

Launch with `CC-Ubuntu24.04` image on the reserved flavor.

In [ ]:
# run in Chameleon Jupyter environment
s = server.Server(
    f"node-proj09-{username}",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name
)
s.submit(idempotent=True)

## Step 3: Associate Floating IP

Assign a public IP so the VM is reachable from the internet.

In [ ]:
# run in Chameleon Jupyter environment
s.associate_floating_ip()

In [ ]:
# run in Chameleon Jupyter environment
s.refresh()
s.show(type="widget")

**Note:** Make a note of the floating IP from the output above (in the "Addresses" row).
You will use it to access the services.

## Step 4: Configure Security Groups

Open ports for SSH, FastAPI, MinIO, Adminer, and Jupyter.

In [ ]:
# run in Chameleon Jupyter environment
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "SSH traffic on TCP port 22"},
    {"name": "allow-8000", "port": 8000, "description": "FastAPI docs on TCP port 8000"},
    {"name": "allow-9001", "port": 9001, "description": "MinIO Console on TCP port 9001"},
    {"name": "allow-5050", "port": 5050, "description": "Adminer UI on TCP port 5050"},
    {"name": "allow-8888", "port": 8888, "description": "Jupyter on TCP port 8888"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Security groups: {[sg['name'] for sg in security_groups]}")

In [ ]:
# run in Chameleon Jupyter environment
s.refresh()
s.check_connectivity()

## Step 5: Retrieve Code on the Instance

Clone the ChatSentry repo onto the VM.

In [ ]:
# run in Chameleon Jupyter environment
s.execute("git clone https://github.com/Rishabhku03/ML_project.git /home/cc/chatsentry 2>/dev/null || cd /home/cc/chatsentry && git pull")

## Step 6: Set Up Docker

Install Docker on the VM and add the user to the docker group.

In [ ]:
# run in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

## Step 7: Open SSH Session

From your **local terminal**, SSH into the VM:

```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>
```

where `<FLOATING_IP>` is the address from Step 3 output.

## Step 8: Set HF_TOKEN and Start Services

Set your HuggingFace token and bring up all containers.

In [ ]:
# run in Chameleon Jupyter environment
# IMPORTANT: Replace 'hf_your_token_here' with your actual HuggingFace token
HF_TOKEN = os.environ.get("HF_TOKEN") or "hf_your_token_here"
s.execute(
    f"echo 'export HF_TOKEN={HF_TOKEN}' >> /home/cc/.bashrc && "
    f"export HF_TOKEN={HF_TOKEN} && "
    f"cd /home/cc/chatsentry/docker && "
    f"docker compose up -d"
)

## Step 9: Verify Services Are Running

Check that all containers are up and healthy.

In [ ]:
# run in Chameleon Jupyter environment
time.sleep(20)  # wait for services to initialize
result = s.execute("docker ps --format 'table {{.Names}}\t{{.Status}}'")
print(result)

## Step 10: Verify Endpoints

Test that the API and MinIO are responding.

In [ ]:
# run in Chameleon Jupyter environment
health_check = s.execute("curl -s http://localhost:8000/health")
print(f"FastAPI /health: {health_check}")

minio_check = s.execute("curl -s -o /dev/null -w '%{http_code}' http://localhost:9001")
print(f"MinIO console: HTTP {minio_check}")

In [ ]:
# run in Chameleon Jupyter environment
# Get the floating IP
import chi
nova = chi.nova()
srv = nova.servers.find(name=f"node-proj09-{username}")
floating_ip = list(srv.addresses.values())[0][-1]['addr']  # extract IP string
print(f"\n{'='*60}")
print("  DEPLOYMENT COMPLETE")
print(f"{'='*60}")
print(f"""
  FastAPI:      http://{floating_ip}:8000/health
  FastAPI docs: http://{floating_ip}:8000/docs
  MinIO:        http://{floating_ip}:9001  (admin / chatsentry_minio)
  Adminer:      http://{floating_ip}:5050  (PostgreSQL / user / chatsentry_pg)
  PostgreSQL:   {floating_ip}:5432

  SSH: ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}
""")

## Step 11: Transfer CSV Dataset (Optional)

The `combined_dataset.csv` (218MB) is too large for GitHub.
Transfer it directly to the VM for ingestion.

**Important:** Run `scp` from your LOCAL terminal (not from inside the SSH session):

```bash
# In your LOCAL terminal (not SSH):
scp -i ~/.ssh/id_rsa_chameleon combined_dataset.csv cc@<FLOATING_IP>:/home/cc/chatsentry/
```

Then go back to your SSH session and run ingestion:

```bash
# On the VM (inside SSH):
cd /home/cc/chatsentry
python3 -m src.data.ingest_and_expand combined_dataset.csv
```

## Teardown

When you are done, delete the VM and lease to release resources.

In [ ]:
# run in Chameleon Jupyter environment
# Uncomment the lines below when you are ready to delete resources:

# s_old = server.get_server(f"node-proj09-{username}")
# s_old.delete()
# print("Server deleted.")

# l = lease.get_lease("proj09_Data")
# lease.delete_lease(l.id)
# print("Lease deleted.")